# Multi-horizon SHARP validation audit

This notebook audits the completed 3-hour and 24-hour SHARP runs. It verifies chronological role reconstruction, label alignment, active-region separation, event separation, and paired active-region block bootstrap intervals. It does **not** train a model, recalibrate probabilities, select a new threshold, or redesign the operational policy.

Labels remain provisional candidate labels. The audit does not establish AIA, GOES or fusion performance.


In [ ]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
assert (ROOT / "scripts/audit_multihorizon_validation.py").is_file(), "Run this notebook from the repository root"


## Run the frozen validation audit

The threshold is read from each already-committed threshold-audit receipt. The 2,000 replicates use `region_component_id` as the resampling unit.


In [ ]:
for horizon in (24, 3):
    cmd = [
        sys.executable,
        "scripts/audit_multihorizon_validation.py",
        "--horizon", str(horizon),
        "--bootstrap-reps", "2000",
        "--bootstrap-seed", "20261007",
    ]
    print("Running", horizon, "hour audit")
    subprocess.run(cmd, check=True)


In [ ]:
import pandas as pd

for horizon in (24, 3):
    out = ROOT / f"results/{horizon}h_graybox/validation_audit_v1"
    print(f"\n--- {horizon} h role metrics ---")
    display(pd.read_csv(out / "role_metrics.csv"))
    print(f"--- {horizon} h AR-block intervals ---")
    display(pd.read_csv(out / "ar_block_bootstrap_intervals.csv"))
    print(f"--- {horizon} h AR overlap ---")
    display(pd.read_csv(out / "region_component_overlap.csv"))
    print(f"--- {horizon} h event overlap ---")
    display(pd.read_csv(out / "event_overlap.csv"))


## Interpretation boundary

Any non-zero training-region overlap must be reported and resolved or sensitivity-tested before describing the evaluation as fully AR-disjoint. Event overlap is reported separately for primary, patch and unresolved event fields. The output receipts are evidence of this audit only; they do not turn provisional labels into confirmatory truth.
